# Akshara: optional Colab GPU run
This notebook has not been executed in Colab. The delivered local pilot was executed on Windows. Upload the project ZIP plus your authorized frozen manifests and images; do not expose tokens or private documents. Choose a GPU runtime first.

In [ ]:
%pip install transformers==5.7.0 peft==0.21.2 accelerate==1.15.0 regex Pillow huggingface_hub safetensors
import torch
assert torch.cuda.is_available(), "Select a GPU runtime"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
from huggingface_hub import notebook_login, snapshot_download
notebook_login()  # account must already have accepted the original model terms
repo=snapshot_download("bodhan-ai/indic-ocr", revision="cd50d301d0e17e8ecb32fc49c8ccbd7914dcfc25")

Extract the project into /content/akshara-ocr and your frozen manifest/image bundle into /content/data. Keep images/word/<hash>.jpg paths matching the manifests. Do not replace the original tokenizer. Use a new run directory.

In [ ]:
%cd /content/akshara-ocr
import subprocess,sys
subprocess.run([sys.executable,"-m","ocr.experiment","probe","--repo",repo,"--data-root","/content/data","--manifest","/content/data/train.jsonl","--limit","4","--steps","64","--output","/content/probe"],check=True)

In [ ]:
subprocess.run([sys.executable,"-m","ocr.experiment","train","--repo",repo,"--data-root","/content/data","--manifest","/content/data/train.jsonl","--validation","/content/data/validation.jsonl","--output","/content/word-training"],check=True)

Freeze model decisions from validation before final testing. Evaluate original and adapted weights on the identical test manifest. Save raw output, metrics, adapter, environment and hashes. Never use test results to choose a new learning rate or epoch count.

In [ ]:
for name,extra in [("baseline",[]),("adapted",["--adapter","/content/word-training/adapter"])]:
    subprocess.run([sys.executable,"-m","ocr.experiment","evaluate","--repo",repo,"--data-root","/content/data","--manifest","/content/data/test.jsonl","--output",f"/content/{name}-test",*extra],check=True)